# 可选实验：Feature Scaling 与 Learning Rate（多变量）


## 目标

在本实验中，你将：

- 使用上一个实验中编写的多变量模型函数。
- 在包含多个 feature（特征）的数据集上运行 Gradient Descent。
- 探索 *learning rate（学习率）$\alpha$* 对 gradient descent 的影响。
- 使用 z-score normalization 进行 *feature scaling（特征缩放）*，改善 gradient descent 的表现。


## 工具

你将使用上一个实验中编写的函数，以及 Matplotlib 和 NumPy。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import  load_house_data, run_gradient_descent 
from lab_utils_multi import  norm_plot, plt_equal_scale, plot_cost_i_w
from lab_utils_common import dlc
np.set_printoptions(precision=2)
plt.style.use('./deeplearning.mplstyle')

## 符号说明

| 符号 | 说明 | Python（如适用） |
| :--- | :--- | :--- |
| $a$ | scalar（标量），不使用粗体 | |
| $\mathbf{a}$ | vector（向量），使用粗体 | |
| $\mathbf{A}$ | matrix（矩阵），使用大写粗体 | |
| **Regression** | | |
| $\mathbf{X}$ | 训练样本 matrix | `X_train` |
| $\mathbf{y}$ | 训练样本的 target（目标值） | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `X[i]`, `y[i]` |
| $m$ | 训练样本数 | `m` |
| $n$ | 每个样本的 feature 数量 | `n` |
| $\mathbf{w}$ | 模型参数：weight（权重） | `w` |
| $b$ | 模型参数：bias（偏置） | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 参数为 $\mathbf{w},b$ 时，模型对 $\mathbf{x}^{(i)}$ 的预测：$f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\mathbf{w}\cdot\mathbf{x}^{(i)}+b$ | `f_wb` |
| $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ | cost 对参数 $w_j$ 的 partial derivative（偏导数），即对应的 gradient 分量 | `dj_dw[j]` |
| $\frac{\partial J(\mathbf{w},b)}{\partial b}$ | cost 对参数 $b$ 的 partial derivative，即对应的 gradient 分量 | `dj_db` |


## 问题描述

与之前的实验一样，本实验以房价预测为例。训练数据集包含多个样本，每个样本有 4 个 feature：面积、卧室数量、楼层数和房龄，如下表所示。注意，本实验的面积单位是 sqft，而更早的一些实验使用的是 1000 sqft。本实验的数据集也比上一个实验更大。

我们希望利用这些数据构建 linear regression 模型，从而预测其他房屋的价格，例如一套面积为 1200 sqft、有 3 间卧室、1 层楼、房龄为 40 年的房屋。

### 数据集

| 面积（sqft） | 卧室数量 | 楼层数 | 房龄（年） | 价格（千美元） |
| --- | --- | --- | --- | --- |
| 952 | 2 | 1 | 65 | 271.5 |
| 1244 | 3 | 2 | 64 | 232 |
| 1947 | 3 | 2 | 17 | 509.8 |
| … | … | … | … | … |


In [ ]:
# load the dataset
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

下面分别绘制各个 feature 与房价的关系图，观察数据集及其 feature。


In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

绘制每个 feature 与 target（房价）的关系，有助于初步判断哪些 feature 与房价的关系更明显。从上图可以看到，面积越大，房价通常越高；卧室数量和楼层数与房价的关系似乎没有那么明显；较新的房屋通常比较旧的房屋价格更高。


<a name="toc_15456_5"></a>
## 多变量 Gradient Descent

下面回顾上一个实验中的多变量 gradient descent 公式。重复以下更新，直到收敛：

$$
\begin{aligned}
w_j &:= w_j-\alpha\frac{\partial J(\mathbf{w},b)}{\partial w_j}, \qquad j=0,\ldots,n-1 \\
b &:= b-\alpha\frac{\partial J(\mathbf{w},b)}{\partial b}
\end{aligned}
\tag{1}
$$

其中，$n$ 是 feature 的数量。所有参数 $w_j$ 和 $b$ 必须**同时更新**，对应的 partial derivative 为：

$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)} \tag{2}
$$

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right) \tag{3}
$$

- $m$ 是数据集中的训练样本数。
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，$y^{(i)}$ 是 target。


## Learning Rate

<figure>
  <img src="./images/C1_W2_Lab06_learningrate.PNG" alt="Learning rate 对参数更新的影响" style="width:1200px;">
</figure>

课程视频讨论了设置 learning rate $\alpha$ 时需要考虑的问题。如公式 (1) 所示，learning rate 控制每次参数更新的幅度，所有参数使用相同的 learning rate。

下面在本数据集上运行 gradient descent，并尝试不同的 $\alpha$ 值。


### $\alpha = 9.9 \times 10^{-7}$


In [ ]:
#set alpha to 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha = 9.9e-7)

Learning rate 似乎过大，算法未能收敛，cost 反而在*增加*。下面绘制结果：


In [ ]:
plot_cost_i_w(X_train, y_train, hist)

右图展示了参数 $w_0$ 的变化。每次迭代时，它都会越过最优值，导致 cost 不断*增加*，而没有接近最小值。

注意，这只是简化的示意图：实际运行时会同时更新 4 个 weight 参数以及 bias，而图中只展示 $w_0$ 的变化，将其他参数固定在适当的值。因此，在本图及后续图中，蓝线和橙线可能会略有偏差。


### $\alpha = 9 \times 10^{-7}$

下面稍微减小 $\alpha$，观察结果。


In [ ]:
#set alpha to 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

运行过程中 cost 持续下降，说明这个 $\alpha$ 值没有过大。


In [ ]:
plot_cost_i_w(X_train, y_train, hist)

左图中，cost 按预期持续下降。右图中，$w_0$ 仍在最小值附近振荡，但每次迭代的 cost 都在下降。注意，当 `w[0]` 越过最优值时，`dj_dw[0]` 会在每次迭代中改变符号。

使用这个 $\alpha$ 值可以收敛。你可以改变迭代次数，观察算法的表现。


### $\alpha = 1 \times 10^{-7}$

下面进一步减小 $\alpha$，观察结果。


In [ ]:
#set alpha to 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

运行过程中 cost 持续下降，说明这个 $\alpha$ 值没有过大。


In [ ]:
plot_cost_i_w(X_train,y_train,hist)

左图中，cost 按预期持续下降。右图中，$w_0$ 没有振荡，而是逐渐接近最小值。整个运行过程中 `dj_w0` 都为负值，这种设置同样可以收敛。


## Feature Scaling

<figure>
  <img src="./images/C1_W2_Lab06_featurescalingheader.PNG" alt="Feature scaling 示意图" style="width:1200px;">
</figure>

课程视频介绍了重新缩放数据集、使各个 feature 具有相近取值范围的重要性。如果你想了解其中的原因，可以展开下面的“详细说明”；随后我们将介绍 feature scaling 的具体实现。


<details>
<summary><span style="color:darkgreen;"><strong>详细说明</strong></span></summary>

再次观察 $\alpha=9\times10^{-7}$ 的情况。这个值已经接近能够避免发散的最大 learning rate。下面展示前几次迭代的结果：

<figure>
  <img src="./images/C1_W2_Lab06_ShortRun.PNG" alt="前几次迭代中的 cost 和参数变化" style="width:1200px;">
</figure>

虽然 cost 在下降，但由于 $w_0$ 对应的 gradient 大得多，它比其他参数变化得更快。

下图展示了使用 $\alpha=9\times10^{-7}$ 长时间运行的结果，这次运行耗时数小时。

<figure>
  <img src="./images/C1_W2_Lab06_LongRun.PNG" alt="长时间运行后的 cost 和参数变化" style="width:1200px;">
</figure>

可以看到，cost 在最初下降后，下降速度变得很慢。请比较 `w0` 与 `w1`、`w2`、`w3`，以及 `dj_dw0` 与 `dj_dw1`、`dj_dw2`、`dj_dw3`。`w0` 很快就接近最终值，`dj_dw0` 也迅速减小，说明 `w0` 已接近收敛；其他参数则收敛得慢得多。

为什么会这样？可以怎样改善？请看下图：

<figure style="text-align:center;">
  <img src="./images/C1_W2_Lab06_scale.PNG" alt="不同 feature 的尺度导致参数更新速度不同">
</figure>

上图说明了各个 $w_j$ 更新速度不同的原因：

- 所有参数（$w_j$ 和 $b$）的更新使用相同的 $\alpha$。
- 计算 $w_j$ 的更新量时，公共的误差项会乘以对应的 feature；计算 $b$ 的更新量时则不会。
- 各个 feature 的数量级差异很大，因此对应参数的更新速度也不同。本例中，$w_0$ 的 gradient 包含面积 `size(sqft)`，其值通常大于 1000；$w_1$ 的 gradient 包含卧室数量，其值通常为 2–4。

解决方法就是 Feature Scaling。

</details>


课程视频讨论了三种方法：

- **Feature scaling**：对于非负 feature，可以将各个值除以该 feature 的最大值（最大值非零时），将范围缩放到 $[0,1]$。更一般地，可以使用 min-max scaling：$(x-\min)/(\max-\min)$；当 $\max\ne\min$ 时，这会将该 feature 的取值缩放到 $[0,1]$，也适用于含负值的 feature。
- **Mean normalization**：$x_i := \dfrac{x_i-\mu_i}{\max-\min}$。
- **Z-score normalization**：下面将详细介绍。


### Z-score Normalization

经过 z-score normalization 后，每个非恒定 feature 的 mean（均值）为 0，standard deviation（标准差）为 1。

按照下面的公式调整输入值，即可实现 z-score normalization：

$$
x_j^{(i)} := \frac{x_j^{(i)}-\mu_j}{\sigma_j} \tag{4}
$$

其中，$j$ 指定一个 feature，也就是 matrix $\mathbf{X}$ 的一列；$\mu_j$ 是第 $j$ 个 feature 所有值的 mean，$\sigma_j$ 是该 feature 的 standard deviation。

$$
\mu_j=\frac{1}{m}\sum_{i=0}^{m-1}x_j^{(i)} \tag{5}
$$

$$
\sigma_j^2=\frac{1}{m}\sum_{i=0}^{m-1}\left(x_j^{(i)}-\mu_j\right)^2 \tag{6}
$$

> **实现提示：** 必须保存 normalization 时使用的 mean 和 standard deviation。模型训练完成后，通常需要预测训练集中未出现过的房屋价格。对于新的输入 $x$（例如房屋面积和卧室数量），必须先使用训练集上计算得到的 mean 和 standard deviation 进行 normalization，再进行预测。

**实现**


In [ ]:
def zscore_normalize_features(X):
    """
    computes  X, zcore normalized by column
    
    Args:
      X (ndarray (m,n))     : input data, m examples, n features
      
    Returns:
      X_norm (ndarray (m,n)): input normalized by column
      mu (ndarray (n,))     : mean of each feature
      sigma (ndarray (n,))  : standard deviation of each feature
    """
    # find the mean of each column/feature
    mu     = np.mean(X, axis=0)                 # mu will have shape (n,)
    # find the standard deviation of each column/feature
    sigma  = np.std(X, axis=0)                  # sigma will have shape (n,)
    # element-wise, subtract mu for that column from each example, divide by std for that column
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)
 
#check our work
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

下面查看 z-score normalization 的具体步骤。下图逐步展示了数据的变换过程。


In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("unnormalized")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("distribution of features before, during, after normalization")
plt.show()

上图展示了训练集中两个 feature——房龄 `age` 和面积 `size(sqft)`——之间的关系。*图中的两个坐标轴使用相同的比例尺。*

- 左图：未经 normalization。`size(sqft)` 的取值范围和 variance（方差）远大于 `age`。
- 中图：第一步，从每个 feature 中减去对应的 mean，使其以 0 为中心。`age` 的变化不太明显，而 `size(sqft)` 明显移到了 0 附近。
- 右图：第二步，除以对应的 standard deviation。两个 feature 都以 0 为中心，并具有相近的尺度。


下面对数据进行 normalization，并与原始数据比较。


In [ ]:
# normalize the original features
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X_train,axis=0)}")   
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X_norm,axis=0)}")

经过 normalization 后，各列 peak-to-peak range（最大值与最小值之差）之间的差异，从数千倍缩小到了约 2–3 倍。


In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("distribution of features before normalization")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle("distribution of features after normalization")

plt.show()

可以看到，normalization 后的数据在 x 轴上以 0 为中心，大致分布在 $\pm2$ 附近。更关键的是，各个 feature 的取值范围现在比较接近。


下面使用 normalization 后的数据重新运行 gradient descent。

注意，这次使用的 **$\alpha$ 大得多**，这将加快 gradient descent。


In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

经过 feature scaling 后，模型**能更快地得到非常准确的结果！** 在这次相对较短的运行结束时，各参数对应的 gradient 都已经很小。对于使用 normalization 后的 feature 的 regression 模型，learning rate 为 0.1 可以作为一个起点。

下面绘制预测值与 target 的对比图。注意，预测时使用 normalization 后的 feature，绘图时使用原始 feature 值。


In [ ]:
#predict target using normalized features
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # plot predictions and targets versus original features    
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlc["dlorange"], label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("target versus prediction using z-score normalized model")
plt.show()

结果看起来不错。需要注意：

- 当存在多个 feature 时，无法再用一张简单的二维图同时展示结果与所有 feature 的关系。
- 图中的预测值使用 normalization 后的 feature 计算。如果参数是在 normalization 后的训练集上学到的，那么预测时的输入也必须采用相同的 normalization。


**Prediction**

构建模型的目的是预测数据集中未出现过的房屋价格。下面预测一套面积为 1200 sqft、有 3 间卧室、1 层楼、房龄为 40 年的房屋。记住，必须使用训练数据 normalization 时计算得到的 mean 和 standard deviation 处理新输入。


In [ ]:
# First, normalize out example.
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict*1000:0.0f}")

**Cost Contours**

<img align="left" src="./images/C1_W2_Lab06_contours.PNG" alt="Feature scaling 前后的 cost contours" style="width:240px;">

还可以通过 cost contours（代价等高线）理解 feature scaling。当 feature 的尺度差异较大时，以模型参数为坐标绘制的 cost contour plot 会非常狭长。

下方各图的参数坐标轴使用相同的比例尺。左图展示了 normalization 之前，面积对应的参数 `w[0]` 与卧室数量对应的参数 `w[1]` 的 cost contours。等高线非常狭长，以至于看不到使其闭合的部分。经过 feature normalization 后，等高线明显更加对称，因此 gradient descent 更新时，各个参数都能更均衡地接近最优值。


In [ ]:
plt_equal_scale(X_train, X_norm, y_train)

## 恭喜完成实验！

在本实验中，你：

- 使用了之前编写的、支持多个 feature 的 linear regression 函数。
- 探索了 learning rate $\alpha$ 对收敛的影响。
- 理解了使用 z-score normalization 进行 feature scaling 如何加快收敛。


## 致谢

本实验的房屋数据来源于 Dean De Cock 为数据科学教学整理的 [Ames Housing dataset](http://jse.amstat.org/v19n3/decock.pdf)。
